# 02 — Data Cleaning

This notebook:
1. Loads raw ridership, station, and weather data
2. Filters invalid/extreme trips
3. Aggregates trips into **hourly pickup counts per station**
4. Fills implicit zeros (station×hour grid)
5. Cleans weather data
6. Saves processed data to `data/processed/`

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_DIR = Path("..").resolve()
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Load raw data
rides = pd.read_csv(RAW_DIR / "rides_raw.csv", parse_dates=["Start_Time", "End_Time"])
stations = pd.read_csv(RAW_DIR / "stations.csv")
weather = pd.read_csv(RAW_DIR / "weather_hourly.csv", parse_dates=["datetime"])

print(f"Rides: {len(rides):,}")
print(f"Stations: {len(stations)}")
print(f"Weather hours: {len(weather):,}")

Rides: 7,083,646
Stations: 1016
Weather hours: 7,296


## 1. Filter Invalid Trips

In [2]:
original_count = len(rides)
print(f"Before filtering: {original_count:,} rides")

# Remove rows with missing station IDs
rides = rides.dropna(subset=["Start_Station_Id", "Start_Time"])

# Filter extreme durations: < 1 min or > 24 hrs
rides["Trip_Duration"] = pd.to_numeric(rides["Trip_Duration"], errors="coerce")
rides = rides[
    (rides["Trip_Duration"] >= 60) &  # at least 1 minute (seconds)
    (rides["Trip_Duration"] <= 86400)  # at most 24 hours
]

# Ensure Start_Time is datetime
rides["Start_Time"] = pd.to_datetime(rides["Start_Time"])

print(f"After filtering: {len(rides):,} rides")
print(f"Removed: {original_count - len(rides):,} rides")

Before filtering: 7,083,646 rides
After filtering: 7,067,741 rides
Removed: 15,905 rides


## 2. Aggregate to Hourly Pickup Counts

In [3]:
# Extract date-hour and station for grouping
rides["hour"] = rides["Start_Time"].dt.floor("h")
rides["station_id"] = rides["Start_Station_Id"].astype(int)

# Count pickups per station per hour
hourly_pickups = (
    rides.groupby(["station_id", "hour"])
    .size()
    .reset_index(name="pickups")
)

print(f"Hourly pickup records (non-zero only): {len(hourly_pickups):,}")
print(f"Stations: {hourly_pickups['station_id'].nunique()}")
print(f"Date range: {hourly_pickups['hour'].min()} → {hourly_pickups['hour'].max()}")
print(f"\nPickup distribution:")
print(hourly_pickups["pickups"].describe())

Hourly pickup records (non-zero only): 2,232,890
Stations: 1031
Date range: 2025-01-01 00:00:00 → 2025-10-31 23:00:00

Pickup distribution:
count    2.232890e+06
mean     3.165288e+00
std      3.911228e+00
min      1.000000e+00
25%      1.000000e+00
50%      2.000000e+00
75%      4.000000e+00
max      1.930000e+02
Name: pickups, dtype: float64


## 3. Fill Implicit Zeros
Create a full (station × hour) grid so hours with zero pickups are explicitly represented.

In [4]:
# Only include stations that appear in both ridership and metadata
valid_stations = sorted(
    set(hourly_pickups["station_id"].unique()) & set(stations["station_id"].unique())
)
print(f"Stations with both ridership and metadata: {len(valid_stations)}")

# Build full hour range
date_range = pd.date_range(
    start=hourly_pickups["hour"].min(),
    end=hourly_pickups["hour"].max(),
    freq="h"
)
print(f"Hour range: {date_range[0]} → {date_range[-1]} ({len(date_range):,} hours)")

# Create full grid
full_grid = pd.MultiIndex.from_product(
    [valid_stations, date_range],
    names=["station_id", "hour"]
).to_frame(index=False)

print(f"Full grid size: {len(full_grid):,} rows")

# Merge with actual pickups, fill missing with 0
hourly_full = full_grid.merge(hourly_pickups, on=["station_id", "hour"], how="left")
hourly_full["pickups"] = hourly_full["pickups"].fillna(0).astype(int)

print(f"Zero-pickup hours: {(hourly_full['pickups'] == 0).sum():,} ({(hourly_full['pickups'] == 0).mean():.1%})")
print(f"Non-zero hours: {(hourly_full['pickups'] > 0).sum():,}")

Stations with both ridership and metadata: 982
Hour range: 2025-01-01 00:00:00 → 2025-10-31 23:00:00 (7,296 hours)
Full grid size: 7,164,672 rows
Zero-pickup hours: 4,988,819 (69.6%)
Non-zero hours: 2,175,853


## 4. Clean Weather Data

In [5]:
# Check for missing values
print("Weather nulls before cleaning:")
print(weather.isnull().sum())

# Interpolate missing values (linear)
weather["temperature_c"] = weather["temperature_c"].interpolate(method="linear")
weather["precipitation_mm"] = weather["precipitation_mm"].fillna(0)
weather["wind_speed_kmh"] = weather["wind_speed_kmh"].interpolate(method="linear")
weather["weather_code"] = weather["weather_code"].ffill()

print("\nWeather nulls after cleaning:")
print(weather.isnull().sum())

# Rename datetime to hour for merging
weather = weather.rename(columns={"datetime": "hour"})

Weather nulls before cleaning:
datetime            0
temperature_c       0
precipitation_mm    0
wind_speed_kmh      0
weather_code        0
dtype: int64

Weather nulls after cleaning:
datetime            0
temperature_c       0
precipitation_mm    0
wind_speed_kmh      0
weather_code        0
dtype: int64


## 5. Save Processed Data

In [6]:
hourly_full.to_csv(PROCESSED_DIR / "hourly_pickups.csv", index=False)
weather.to_csv(PROCESSED_DIR / "weather_clean.csv", index=False)
stations.to_csv(PROCESSED_DIR / "stations_clean.csv", index=False)

print("Saved to data/processed/:")
for f in sorted(PROCESSED_DIR.iterdir()):
    size_mb = f.stat().st_size / 1e6
    print(f"  {f.name} — {size_mb:.1f} MB")

Saved to data/processed/:
  hourly_pickups.csv — 200.7 MB
  stations_clean.csv — 0.1 MB
  weather_clean.csv — 0.3 MB
